In [1]:
print("hello")

hello


In [2]:
# ============================================================
# HYBRID CYBER-PHYSICAL SECURITY CLASSIFICATION
# Current Synthetic Dataset
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


# ============================================================
# 1. LOAD DATA
# ============================================================

FILE = "/content/synthetic_cyber_physical_dataset_800.xlsx"

users = pd.read_excel(FILE, sheet_name="Users")
physical = pd.read_excel(FILE, sheet_name="Physical Events")
cyber = pd.read_excel(FILE, sheet_name="Cyber Events")


print("Users:", users.shape)
print("Physical:", physical.shape)
print("Cyber:", cyber.shape)



Users: (50, 6)
Physical: (800, 6)
Cyber: (800, 9)


In [3]:

# ============================================================
# 2. CONVERT TIMESTAMPS
# ============================================================

physical["timestamp"] = pd.to_datetime(
    physical["timestamp"]
)

cyber["timestamp"] = pd.to_datetime(
    cyber["timestamp"]
)


# ============================================================
# 3. RENAME TIMESTAMP COLUMNS
# ============================================================

physical = physical.rename(
    columns={
        "timestamp": "physical_timestamp"
    }
)

cyber = cyber.rename(
    columns={
        "timestamp": "cyber_timestamp"
    }
)


# ============================================================
# 4. MERGE PHYSICAL + CYBER
# ============================================================
#
# In the current generated dataset:
#
# P0001 corresponds to C0001
# P0002 corresponds to C0002
# ...
#
# We therefore create a common numeric event number.
#
# IMPORTANT:
# This is specific to the current synthetic dataset.
# A future realistic dataset should correlate events using
# user/device/time rather than row number.
# ============================================================

physical["event_number"] = (
    physical["event_id"]
    .str.extract(r"(\d+)")
    .astype(int)
)

cyber["event_number"] = (
    cyber["event_id"]
    .str.extract(r"(\d+)")
    .astype(int)
)


data = physical.merge(
    cyber,
    on=["event_number", "user_id"],
    suffixes=("_physical", "_cyber")
)




In [4]:
# ============================================================
# 5. ADD USER INFORMATION
# ============================================================

data = data.merge(
    users,
    on="user_id",
    how="left"
)


print("\nCombined dataset:")
print(data.shape)





Combined dataset:
(800, 20)


In [5]:
# ============================================================
# 6. DERIVE TIME FEATURES
# ============================================================

data["physical_hour"] = (
    data["physical_timestamp"].dt.hour
)

data["cyber_hour"] = (
    data["cyber_timestamp"].dt.hour
)


# Day of week
# Monday = 0
# Sunday = 6

data["day_of_week"] = (
    data["physical_timestamp"].dt.dayofweek
)


data["is_weekend"] = (
    data["day_of_week"] >= 5
).astype(int)


# ============================================================
# 7. CALCULATE AFTER-HOURS
# ============================================================

# Convert working-hour strings to hours

data["normal_start_hour"] = (
    pd.to_datetime(
        data["normal_start"],
        format="%H:%M"
    ).dt.hour
)

data["normal_end_hour"] = (
    pd.to_datetime(
        data["normal_end"],
        format="%H:%M"
    ).dt.hour
)


data["after_hours"] = (
    (
        data["physical_hour"]
        < data["normal_start_hour"]
    )
    |
    (
        data["physical_hour"]
        >= data["normal_end_hour"]
    )
).astype(int)


# Security Officers have 00:00 - 23:59
# The hour-only calculation can incorrectly classify 23:xx.
# Correct them:

data.loc[
    (
        (data["normal_start"] == "00:00") &
        (data["normal_end"] == "23:59")
    ),
    "after_hours"
] = 0




In [6]:
# ============================================================
# 8. RESTRICTED LOCATION
# ============================================================

data["restricted_location"] = (
    data["location_type"]
    .str.lower()
    .eq("restricted")
    .astype(int)
)


# ============================================================
# 9. ACCESS RESULT
# ============================================================

data["access_denied"] = (
    data["access_result"]
    .str.lower()
    .eq("denied")
    .astype(int)
)


# ============================================================
# 10. CYBER-PHYSICAL TIME DIFFERENCE
# ============================================================

data["time_gap_minutes"] = (
    (
        data["cyber_timestamp"]
        - data["physical_timestamp"]
    )
    .dt.total_seconds()
    / 60
)


# Absolute value for safety

data["time_gap_minutes"] = (
    data["time_gap_minutes"].abs()
)


# ============================================================
# 11. ENCODE CYBER SEVERITY
# ============================================================

severity_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

data["severity_numeric"] = (
    data["severity"]
    .map(severity_mapping)
)


# ============================================================
# 12. CREATE A TEMPORARY RISK SCORE
# ============================================================
#
# IMPORTANT:
# This is NOT ML.
#
# It creates provisional labels because the current dataset
# does not contain independent Low/Medium/High ground truth.
#
# You should document these assumptions.
# ============================================================

risk_score = np.zeros(len(data))


# -------------------------
# Physical evidence
# -------------------------

risk_score += (
    data["after_hours"] * 1
)

risk_score += (
    data["restricted_location"] * 1
)

risk_score += (
    data["access_denied"] * 1
)


# -------------------------
# Cyber evidence
# -------------------------

risk_score += (
    data["severity_numeric"]
)

risk_score += (
    (data["failed_logins"] >= 5).astype(int)
)

risk_score += (
    (data["failed_logins"] >= 10).astype(int)
)

risk_score += (
    (data["sensitive_files"] >= 10).astype(int)
)

risk_score += (
    (data["usb_connected"] == 1).astype(int)
)


# -------------------------
# Temporal correlation
# -------------------------

risk_score += (
    (data["time_gap_minutes"] <= 10)
    .astype(int)
)


data["risk_score"] = risk_score




In [7]:
# ============================================================
# 13. CONVERT SCORE INTO RISK LEVEL
# ============================================================

def assign_risk(score):

    if score <= 2:
        return "Low"

    elif score <= 5:
        return "Medium"

    else:
        return "High"


data["risk_level"] = (
    data["risk_score"]
    .apply(assign_risk)
)


print("\nRisk distribution:")
print(
    data["risk_level"]
    .value_counts()
)




Risk distribution:
risk_level
Low       456
Medium    217
High      127
Name: count, dtype: int64


In [8]:
data

,event_id_physical,user_id,physical_timestamp,location,location_type,access_result,event_number,event_id_cyber,device_id_x,cyber_timestamp,...,is_weekend,normal_start_hour,normal_end_hour,after_hours,restricted_location,access_denied,time_gap_minutes,severity_numeric,risk_score,risk_level
0,P0001,U050,2026-04-06 03:44:30,Research Lab,Restricted,Granted,1,C0001,D050,2026-04-06 03:54:55,...,0,8,18,1,1,0,10.416667,1,3.0,Medium
1,P0002,U045,2026-03-17 09:42:37,Meeting Room,Normal,Granted,2,C0002,D045,2026-03-17 09:57:21,...,0,0,23,0,0,0,14.733333,0,0.0,Low
2,P0003,U030,2026-03-19 04:16:14,Server Room,Restricted,Denied,3,C0003,D030,2026-03-19 04:24:26,...,0,8,17,1,1,1,8.200000,0,4.0,Medium
3,P0004,U014,2026-04-18 15:17:46,Meeting Room,Normal,Granted,4,C0004,D014,2026-04-18 15:29:46,...,1,8,17,0,0,0,12.000000,2,4.0,Medium
4,P0005,U046,2026-04-05 02:17:02,Server Room,Restricted,Denied,5,C0005,D046,2026-04-05 02:23:57,...,1,8,17,1,1,1,6.916667,2,9.0,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,P0796,U026,2026-03-17 08:07:15,Cafeteria,Normal,Granted,796,C0796,D026,2026-03-17 08:31:57,...,0,8,18,0,0,0,24.700000,0,1.0,Low
796,P0797,U031,2026-04-20 22:24:17,Research Lab,Restricted,Granted,797,C0797,D031,2026-04-20 22:30:48,...,0,8,18,1,1,0,6.516667,1,4.0,Medium
797,P0798,U042,2026-04-13 16:14:01,Cafeteria,Normal,Granted,798,C0798,D042,2026-04-13 16:37:45,...,0,8,18,0,0,0,23.733333,0,0.0,Low
798,P0799,U028,2026-03-21 11:06:20,General Office,Normal,Granted,799,C0799,D028,2026-03-21 11:27:12,...,1,9,17,0,0,0,20.866667,0,0.0,Low


In [9]:
data.to_csv("output.csv", index=False)